In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
# ============================================================
# CELL 0 — GPU sanity check
# ============================================================
import torch
print("CUDA available :", torch.cuda.is_available())
print("GPU count       :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}         : {torch.cuda.get_device_name(i)}")

CUDA available : True
GPU count       : 2
  GPU 0         : Tesla T4
  GPU 1         : Tesla T4


In [3]:
!git clone https://github.com/THU-MIG/RepViT.git
%cd RepViT

Cloning into 'RepViT'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 374 (delta 81), reused 64 (delta 64), pack-reused 272 (from 1)
Receiving objects: 100% (374/374), 20.74 MiB | 31.70 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/kaggle/working/RepViT


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
# ============================================================
# CELL 1 — Install & imports
# ============================================================
import subprocess, sys

subprocess.run(["pip", "install", "timm", "--quiet"], check=True)

import os
if not os.path.exists("/kaggle/working/RepViT"):
    subprocess.run(["git", "clone", "https://github.com/THU-MIG/RepViT.git",
                    "/kaggle/working/RepViT"], check=True)
    subprocess.run(["pip", "install", "-r", "/kaggle/working/RepViT/requirements.txt",
                    "--quiet"], check=True)

sys.path.insert(0, "/kaggle/working/RepViT")

import math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import pandas as pd

print("All imports OK")
print("RepViT folder:", os.listdir("/kaggle/working/RepViT"))

All imports OK
RepViT folder: ['data', 'eval.sh', 'README.md', 'utils.py', 'LICENSE', 'engine.py', 'model', 'logs', 'flops.py', '.git', 'requirements.txt', 'figures', 'sam', 'segmentation', 'main.py', 'speed_gpu.py', 'detection', 'export_coreml.py', 'train.sh', 'losses.py', '.gitignore']


In [47]:
# ============================================================
# FIXED CONFIG (Aligned with improved training strategy)
# ============================================================

IMAGENET_DIR = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC"

# Symlinked paths (make sure you created them)
TRAIN_DIR = "/kaggle/working/imagenet/train"
VAL_DIR   = "/kaggle/working/imagenet/val"

# Training params
BATCH_SIZE   = 64          # safer + better generalization
NUM_WORKERS  = 2           # Kaggle stable
NUM_CLASSES  = 1000

# Training phases
EPOCHS_HEAD      = 5       # freeze backbone
EPOCHS_FINETUNE  = 10      # unfreeze

# Learning rates (IMPORTANT)
LR_HEAD      = 1e-4
LR_BACKBONE  = 1e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device         : {DEVICE}")
print(f"GPUs           : {torch.cuda.device_count()}")
print(f"Train dir      : {TRAIN_DIR}")
print(f"Val dir        : {VAL_DIR}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Epochs (head)  : {EPOCHS_HEAD}")
print(f"Epochs (fine)  : {EPOCHS_FINETUNE}")
print(f"LR backbone    : {LR_BACKBONE}")
print(f"LR head        : {LR_HEAD}")

Device         : cuda
GPUs           : 2
Train dir      : /kaggle/working/imagenet/train
Val dir        : /kaggle/working/imagenet/val
Batch size     : 64
Epochs (head)  : 5
Epochs (fine)  : 10
LR backbone    : 1e-05
LR head        : 0.0001


In [10]:
# ============================================================
# CELL 3 — Build 150k symlink training set
# 150 images per class × 1000 classes = 150 000 total
# (was 50/class = 50k — too small, hurt accuracy)
# ============================================================
if not os.path.exists(TRAIN_DIR):
    print("Building 150k symlinks...")
    t = time.time()
    train_path = os.path.join(IMAGENET_DIR, "train")
    classes = sorted(os.listdir(train_path))
    np.random.seed(42)
    count = 0
    for cls in classes:
        cls_src = os.path.join(train_path, cls)
        cls_dst = os.path.join(TRAIN_DIR, cls)
        os.makedirs(cls_dst, exist_ok=True)
        images = os.listdir(cls_src)
        pick = np.random.choice(images, min(150, len(images)), replace=False)
        for img in pick:
            src = os.path.join(cls_src, img)
            dst = os.path.join(cls_dst, img)
            if not os.path.exists(dst):
                os.symlink(src, dst)
            count += 1
    print(f"Done! {count} symlinks in {time.time()-t:.1f}s")
else:
    count = sum(len(f) for _, _, f in os.walk(TRAIN_DIR))
    print(f"Already exists — {count} files in TRAIN_DIR")

Building 150k symlinks...
Done! 150000 symlinks in 44.0s


In [11]:
# ============================================================
# Build organised val set (ROBUST VERSION)
# ============================================================

import os, time
import pandas as pd

SOL_CSV = "/kaggle/input/competitions/imagenet-object-localization-challenge/LOC_val_solution.csv"

if not os.path.exists(VAL_DIR):
    print("Organising val by class...")
    t = time.time()

    sol = pd.read_csv(SOL_CSV)

    # Extract class label
    sol["class"] = sol["PredictionString"].str.split().str[0]

    val_path = os.path.join(IMAGENET_DIR, "val")

    total_created = 0
    total_skipped = 0

    for _, row in sol.iterrows():
        img_name = row["ImageId"] + ".JPEG"
        cls = row["class"]

        src = os.path.join(val_path, img_name)
        dst_dir = os.path.join(VAL_DIR, cls)
        dst = os.path.join(dst_dir, img_name)

        if not os.path.exists(src):
            total_skipped += 1
            continue

        os.makedirs(dst_dir, exist_ok=True)

        if not os.path.exists(dst):
            try:
                os.symlink(src, dst)
                total_created += 1
            except Exception:
                total_skipped += 1

    print(f"Done! {total_created} symlinks created in {time.time()-t:.1f}s")
    print(f"Skipped: {total_skipped}")

else:
    count = sum(len(files) for _, _, files in os.walk(VAL_DIR))
    print(f"Already exists — {count} val images")

Organising val by class...
Done! 50000 symlinks created in 208.0s
Skipped: 0


In [12]:
# Total val images
total = sum(len(files) for _, _, files in os.walk(VAL_DIR))
print("Total val images:", total)

# Check one class
sample_class = os.listdir(VAL_DIR)[0]
print(sample_class, len(os.listdir(os.path.join(VAL_DIR, sample_class))))

Total val images: 50000
n02102040 50


In [15]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [20]:
import os, time, torch, numpy as np, pandas as pd
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm import create_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [23]:
# ============================================================
# CELL 6 — RepViT Manual Load + SCE Head Injection
# ============================================================

import os
import torch
import torch.nn as nn
import urllib.request
from model import repvit


# ============================================================
# Selective Channel Enhancement Head (OUR CONTRIBUTION)
# ============================================================
class SCEHead(nn.Module):
    """
    BatchNorm1d → SE-style channel recalibration → Dropout → Linear
    """
    def __init__(self, in_features: int, num_classes: int,
                 reduction: int = 8, dropout: float = 0.2):
        super().__init__()

        mid = max(in_features // reduction, 16)

        self.bn = nn.BatchNorm1d(in_features)

        self.se = nn.Sequential(
            nn.Linear(in_features, mid, bias=False),
            nn.GELU(),
            nn.Linear(mid, in_features, bias=False),
            nn.Sigmoid()
        )

        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(in_features, num_classes)

        nn.init.trunc_normal_(self.fc.weight, std=0.02)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = self.bn(x)
        x = x * self.se(x)
        x = self.drop(x)
        return self.fc(x)


# ============================================================
# Load RepViT manually
# ============================================================
WEIGHTS_URL = "https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m0_9_distill_300e.pth"
WEIGHTS_PATH = "/kaggle/working/repvit_m0_9.pth"

print("Building RepViT-m0.9 backbone...")
model = repvit.repvit_m0_9(pretrained=False)

if not os.path.exists(WEIGHTS_PATH):
    print("Downloading pretrained weights...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print("Download complete.")
else:
    print("Weights already cached.")


checkpoint = torch.load(WEIGHTS_PATH, map_location="cpu")
state_dict = checkpoint.get("model", checkpoint.get("state_dict", checkpoint))

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print(f"Missing keys   : {len(missing)}")
print(f"Unexpected keys: {len(unexpected)} (normal for distillation)")


# ============================================================
# Extract classifier info
# ============================================================
print("\nOriginal classifier:", model.classifier)

bn_linear = model.classifier.classifier
in_features = bn_linear.l.in_features
num_out = bn_linear.l.out_features

print("Feature size:", in_features)
print("Original classes:", num_out)


# ============================================================
# Build SCE Head
# ============================================================
sce = SCEHead(
    in_features=in_features,
    num_classes=NUM_CLASSES,
    reduction=8,
    dropout=0.2
)


# ============================================================
# Copy pretrained BN + Linear weights (IMPORTANT)
# ============================================================
sce.bn.weight.data.copy_(bn_linear.bn.weight.data)
sce.bn.bias.data.copy_(bn_linear.bn.bias.data)
sce.bn.running_mean.copy_(bn_linear.bn.running_mean)
sce.bn.running_var.copy_(bn_linear.bn.running_var)
sce.bn.num_batches_tracked.copy_(bn_linear.bn.num_batches_tracked)

sce.fc.weight.data.copy_(bn_linear.l.weight.data)
sce.fc.bias.data.copy_(bn_linear.l.bias.data)

print("Pretrained BN + FC weights copied ✔")


# ============================================================
# Replace classifier with SCE head
# ============================================================
model.classifier.classifier = sce

print("\nClassifier successfully replaced with SCE ✔")


# ============================================================
# Parameter analysis
# ============================================================
total_params = sum(p.numel() for p in model.parameters()) / 1e6
head_params = sum(p.numel() for p in model.classifier.classifier.parameters()) / 1e6

print(f"\nTotal params : {total_params:.2f}M")
print(f"SCE head     : {head_params:.4f}M ({head_params/total_params*100:.2f}%)")


# ============================================================
# Move to device + optional multi-GPU
# ============================================================
if torch.cuda.device_count() > 1:
    print(f"Using DataParallel on {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(DEVICE)


# ============================================================
# Sanity check
# ============================================================
model.eval()
with torch.no_grad():
    imgs, labs = next(iter(val_loader))
    imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)

    preds = model(imgs).argmax(1)
    acc = (preds == labs).float().mean().item()

print(f"\nQuick val accuracy (1 batch): {acc:.4f}")
print("Model loaded + SCE injected successfully ✔")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/kaggle/working/RepViT/model/repvit.py:276: UserWarning: Overwriting repvit_m0_9 in registry with model.repvit.repvit_m0_9. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/RepViT/model/repvit.py:312: UserWarning: Overwriting repvit_m1_0 in registry with model.repvit.repvit_m1_0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/RepViT/model/repvit.py:349: UserWarning: Overwriting repvit_m1_1 in registry with model.repvit.repvit_m1_1. This is because the name being registered conflicts with an existing name. Please check if 

Building RepViT-m0.9 backbone...
Download complete.
Missing keys   : 0
Unexpected keys: 7 (normal for distillation)

Original classifier: Classfier(
  (classifier): BN_Linear(
    (bn): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (l): Linear(in_features=384, out_features=1000, bias=True)
  )
)
Feature size: 384
Original classes: 1000
Pretrained BN + FC weights copied ✔

Classifier successfully replaced with SCE ✔

Total params : 5.14M
SCE head     : 0.4226M (8.22%)
Using DataParallel on 2 GPUs

Quick val accuracy (1 batch): 0.9219
Model loaded + SCE injected successfully ✔


In [36]:
net = model.module if isinstance(model, nn.DataParallel) else model

for name, param in net.named_parameters():
    # freeze everything except classifier (and SCE inside it)
    if "classifier" not in name:
        param.requires_grad = False

print("Backbone frozen ✔ (classifier + SCE trainable)")

Backbone frozen ✔ (classifier + SCE trainable)


In [37]:
net = model.module if isinstance(model, nn.DataParallel) else model

# freeze everything
for param in net.parameters():
    param.requires_grad = False

# unfreeze classifier (includes SCE)
for param in net.classifier.parameters():
    param.requires_grad = True

print("Only classifier + SCE are trainable ✔")

Only classifier + SCE are trainable ✔


In [39]:
net = model.module if isinstance(model, nn.DataParallel) else model

print(net.classifier)
print("\n--- SCE MODULE ---")
print(net.classifier.classifier)

Classfier(
  (classifier): SCEHead(
    (bn): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (se): Sequential(
      (0): Linear(in_features=384, out_features=48, bias=False)
      (1): GELU(approximate='none')
      (2): Linear(in_features=48, out_features=384, bias=False)
      (3): Sigmoid()
    )
    (drop): Dropout(p=0.2, inplace=False)
    (fc): Linear(in_features=384, out_features=1000, bias=True)
  )
)

--- SCE MODULE ---
SCEHead(
  (bn): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (se): Sequential(
    (0): Linear(in_features=384, out_features=48, bias=False)
    (1): GELU(approximate='none')
    (2): Linear(in_features=48, out_features=384, bias=False)
    (3): Sigmoid()
  )
  (drop): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=384, out_features=1000, bias=True)
)


In [40]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

In [27]:
net = model.module if isinstance(model, nn.DataParallel) else model
num_features = net.classifier.classifier.fc.in_features
print(num_features)

384


In [42]:
train_losses = []
val_losses = []

train_accs = []
val_accs = []

In [44]:
def evaluate(model):
    model.eval()

    correct = total = 0
    loss_sum = 0

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            out = model(x)
            loss = criterion(out, y)

            loss_sum += loss.item()

            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = 100 * correct / total
    avg_loss = loss_sum / len(val_loader)

    return acc, avg_loss

In [ ]:
for epoch in range(3):   # short warm-up phase

    model.train()
    correct = total = 0
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred = out.argmax(1)

        correct += (pred == y).sum().item()
        total += y.size(0)

    train_acc = correct / total
    val_acc, val_loss = evaluate(model)

    print(f"[Phase 1] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Train Loss: {train_loss:.4f}")
    print(f"Val Acc  : {val_acc:.2f}% | Val Loss  : {val_loss:.4f}")

    print(f"[Phase 1] Epoch {epoch+1}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.2f}%")

In [ ]:
for param in net.backbone.parameters():
    param.requires_grad = True

print("Backbone unfrozen ✔ (Full fine-tuning enabled)")

In [ ]:
best_acc = 0

for epoch in range(EPOCHS):

    model.train()

    train_correct = 0
    train_total = 0
    train_loss_sum = 0

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()

        pred = out.argmax(1)
        train_correct += (pred == y).sum().item()
        train_total += y.size(0)

    # ---- TRAIN METRICS ----
    train_acc = 100 * train_correct / train_total
    train_loss = train_loss_sum / len(train_loader)

    # ---- VALIDATION ----
    val_acc, val_loss = evaluate(model)

    scheduler.step()

    # ---- STORE METRICS ----
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Acc: {train_acc:.2f}% | Train Loss: {train_loss:.4f}")
    print(f"Val Acc  : {val_acc:.2f}% | Val Loss  : {val_loss:.4f}")

    # ---- CHECKPOINT SAVE ----
    if val_acc > best_acc:
        best_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc": val_acc,
        }, "best_sce_checkpoint.pth")

        print("Best checkpoint saved ✔")

In [ ]:
torch.save(model.state_dict(), "final_sce_model.pth")
print("Final model saved ✔")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        out = model(x).argmax(1).cpu().numpy()

        all_preds.extend(out)
        all_labels.extend(y.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

final_acc = (np.array(all_preds) == np.array(all_labels)).mean() * 100
print(f"\nFINAL ACCURACY: {final_acc:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")

plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure()

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()